# Assignment No. 5 — Generative AI (AI4009, Spring 2026)
## Vision Language Models (VLM) using QLoRA Fine-Tuning for Document-to-Markdown Generation
**Model:** Qwen2-VL-2B-Instruct | **Platform:** Kaggle GPU T4 x2 | **Student ID:** 22F-8816

---

## 🔧 Section 0 — Environment Setup & Installs

In [ ]:
# Install required libraries
!pip install -q transformers==4.45.0 peft==0.12.0 bitsandbytes==0.44.0 \
    accelerate==0.34.0 datasets trl einops qwen-vl-utils \
    torchvision pillow rouge-score nltk gradio

!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo 'flash-attn not installed (optional)'

In [ ]:
import os, re, json, random, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import torch
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}  : {torch.cuda.get_device_name(i)}')

## 📁 Part 1 — Dataset Exploration

In [ ]:
# ── Paths (Kaggle input directory) ───────────────────────────────────────────
DATASET_ROOT = Path('/kaggle/input/nougat-training-dataset-example')

# List top-level contents
print('Dataset root contents:')
for p in sorted(DATASET_ROOT.iterdir()):
    print(f'  {p.name}  ({"dir" if p.is_dir() else "file"})')

In [ ]:
# ── Discover image-markdown pairs ────────────────────────────────────────────
image_extensions = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}

all_images = sorted(DATASET_ROOT.rglob('*'))
all_images = [p for p in all_images if p.suffix.lower() in image_extensions]

pairs = []
for img_path in all_images:
    # Look for matching .mmd or .md file (same stem)
    for ext in ['.mmd', '.md', '.txt']:
        md_path = img_path.with_suffix(ext)
        if not md_path.exists():
            # try sibling directory named 'markdown' or 'labels'
            alt = img_path.parent.parent / 'markdown' / (img_path.stem + ext)
            if alt.exists():
                md_path = alt
            else:
                md_path = None
        if md_path and md_path.exists():
            pairs.append({'image': img_path, 'markdown': md_path})
            break

print(f'Total images found   : {len(all_images)}')
print(f'Paired image-md sets : {len(pairs)}')

In [ ]:
# ── Visual inspection of 3 random pairs ──────────────────────────────────────
sample_pairs = random.sample(pairs, min(3, len(pairs)))

for i, pair in enumerate(sample_pairs):
    img  = Image.open(pair['image']).convert('RGB')
    md   = pair['markdown'].read_text(encoding='utf-8', errors='ignore')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(img)
    axes[0].set_title(f'Sample {i+1} — Document Image', fontsize=12)
    axes[0].axis('off')

    axes[1].text(0.01, 0.99, md[:1200], transform=axes[1].transAxes,
                 fontsize=7, verticalalignment='top', family='monospace',
                 wrap=True)
    axes[1].set_title(f'Sample {i+1} — Ground Truth Markdown', fontsize=12)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    print(f'Image size : {img.size}   |  Markdown length : {len(md)} chars')
    print('-' * 70)

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────────
md_lengths  = [len(p['markdown'].read_text(errors='ignore')) for p in pairs]
img_widths  = [Image.open(p['image']).size[0] for p in pairs]
img_heights = [Image.open(p['image']).size[1] for p in pairs]

df_stats = pd.DataFrame({
    'markdown_chars': md_lengths,
    'img_width'     : img_widths,
    'img_height'    : img_heights,
})
print(df_stats.describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(md_lengths,  bins=30, color='steelblue',  edgecolor='white')
axes[0].set_title('Markdown Length Distribution')
axes[1].hist(img_widths,  bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Image Width Distribution')
axes[2].hist(img_heights, bins=30, color='seagreen',   edgecolor='white')
axes[2].set_title('Image Height Distribution')
for ax in axes:
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Nougat Dataset Statistics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🗂️ Part 2 — Data Preparation (ChatML Format)

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
MODEL_ID   = 'Qwen/Qwen2-VL-2B-Instruct'
IMG_SIZE   = 640     # resize longest edge to this before feeding the model
MAX_TOKENS = 1024    # max new tokens to generate

INSTRUCTION = (
    'You are an expert document parser. '
    'Convert the document image below into well-structured Markdown. '
    'Preserve headings, lists, tables, equations, and code blocks exactly. '
    'Output ONLY the Markdown text without any explanation.'
)

def build_chatml_sample(image_path: Path, markdown_text: str) -> dict:
    """Build a single ChatML-format sample."""
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': str(image_path)},
                    {'type': 'text',  'text': INSTRUCTION},
                ],
            },
            {
                'role': 'assistant',
                'content': markdown_text.strip(),
            },
        ]
    }

# Build dataset list
dataset_list = []
for pair in tqdm(pairs, desc='Building ChatML samples'):
    md_text = pair['markdown'].read_text(encoding='utf-8', errors='ignore').strip()
    if len(md_text) < 10:   # skip near-empty markdowns
        continue
    dataset_list.append(build_chatml_sample(pair['image'], md_text))

print(f'Total valid ChatML samples: {len(dataset_list)}')
print('\nExample sample structure:')
sample = dataset_list[0]
print(f"  Role: {sample['messages'][0]['role']}")
print(f"  Content types: {[c['type'] for c in sample['messages'][0]['content']]}")
print(f"  Assistant response (first 200 chars): {sample['messages'][1]['content'][:200]}")

## ✂️ Part 3 — Dataset Splitting (80 / 20)

In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    dataset_list, test_size=0.20, random_state=SEED
)

print(f'Training   samples : {len(train_data)}')
print(f'Validation samples : {len(val_data)}')

# Quick pie chart
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [len(train_data), len(val_data)],
    labels=['Train (80%)', 'Validation (20%)'],
    autopct='%1.1f%%',
    colors=['steelblue', 'coral'],
    startangle=90,
)
ax.set_title('Dataset Split', fontsize=13)
plt.tight_layout()
plt.show()

## 🤖 Part 4 — QLoRA Fine-Tuning

### 4.1 — Load Tokenizer & Processor

In [ ]:
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ── 4-bit quantization config ─────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

# ── Load processor ────────────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = 'right'

print('Processor loaded ✓')

### 4.2 — Load Base Model in 4-bit

In [ ]:
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config  = bnb_config,
    device_map           = 'auto',         # auto-spreads across both T4 GPUs
    trust_remote_code    = True,
    torch_dtype          = torch.bfloat16,
)

# Prepare for k-bit training (freezes BN, casts LM head to fp32, etc.)
base_model = prepare_model_for_kbit_training(base_model)

print('Base model loaded in 4-bit ✓')
print(f'Parameters : {sum(p.numel() for p in base_model.parameters())/1e6:.1f}M')

### 4.3 — Attach LoRA Adapters

In [ ]:
# Identify attention/MLP projection layers to target
TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

lora_config = LoraConfig(
    r              = 16,           # LoRA rank
    lora_alpha     = 32,           # scaling factor
    target_modules = TARGET_MODULES,
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

### 4.4 — Custom Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from qwen_vl_utils import process_vision_info   # Qwen2-VL utility

class NougatVLMDataset(Dataset):
    """PyTorch Dataset wrapping ChatML-format samples."""

    def __init__(self, data: list, processor, img_size: int = 640):
        self.data      = data
        self.processor = processor
        self.img_size  = img_size

    def __len__(self):
        return len(self.data)

    def _resize(self, img: Image.Image) -> Image.Image:
        w, h   = img.size
        scale  = self.img_size / max(w, h)
        new_w  = int(w * scale)
        new_h  = int(h * scale)
        return img.resize((new_w, new_h), Image.LANCZOS)

    def __getitem__(self, idx: int) -> dict:
        sample  = self.data[idx]
        messages = sample['messages']

        # ── Load & resize image ──────────────────────────────────────────────
        img_path = messages[0]['content'][0]['image']
        image    = Image.open(img_path).convert('RGB')
        image    = self._resize(image)

        # Replace path with PIL image so process_vision_info can handle it
        msgs_copy = json.loads(json.dumps(messages))
        msgs_copy[0]['content'][0] = {'type': 'image', 'image': image}

        # ── Build prompt text ────────────────────────────────────────────────
        text = self.processor.apply_chat_template(
            msgs_copy, tokenize=False, add_generation_prompt=False
        )

        # ── Process vision info (pixel values) ───────────────────────────────
        image_inputs, video_inputs = process_vision_info(msgs_copy)

        # ── Tokenise ─────────────────────────────────────────────────────────
        inputs = self.processor(
            text          = [text],
            images        = image_inputs,
            videos        = video_inputs,
            padding       = False,
            return_tensors= 'pt',
        )
        # Squeeze batch dim added by processor
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        # ── Build labels (mask prompt tokens with -100) ───────────────────────
        # Find where assistant turn starts by encoding the assistant prefix
        labels        = inputs['input_ids'].clone()
        # Everything before the last <|im_start|>assistant is masked
        assistant_ids = self.processor.tokenizer.encode(
            '<|im_start|>assistant\n', add_special_tokens=False
        )
        seq_len       = labels.shape[0]
        a_len         = len(assistant_ids)
        mask_end      = seq_len   # fallback: mask all
        for i in range(seq_len - a_len):
            if labels[i: i + a_len].tolist() == assistant_ids:
                mask_end = i + a_len
                break
        labels[:mask_end] = -100

        inputs['labels'] = labels
        return inputs


def collate_fn(batch: list) -> dict:
    """Pad variable-length sequences in a batch."""
    keys   = batch[0].keys()
    result = {}
    for k in keys:
        tensors = [item[k] for item in batch]
        if k == 'labels':
            result[k] = torch.nn.utils.rnn.pad_sequence(
                tensors, batch_first=True, padding_value=-100
            )
        elif tensors[0].dtype in (torch.long, torch.int):
            result[k] = torch.nn.utils.rnn.pad_sequence(
                tensors, batch_first=True,
                padding_value=processor.tokenizer.pad_token_id or 0,
            )
        else:
            # pixel_values or attention_mask
            try:
                result[k] = torch.stack(tensors)
            except Exception:
                result[k] = tensors   # fallback: list
    return result


train_dataset = NougatVLMDataset(train_data, processor, img_size=IMG_SIZE)
val_dataset   = NougatVLMDataset(val_data,   processor, img_size=IMG_SIZE)

print(f'Train dataset size : {len(train_dataset)}')
print(f'Val   dataset size : {len(val_dataset)}')

# Smoke-test one sample
sample = train_dataset[0]
print('Sample keys      :', list(sample.keys()))
print('input_ids shape  :', sample['input_ids'].shape)

### 4.5 — Training Configuration & Loop

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# ── Hyper-parameters ─────────────────────────────────────────────────────────
NUM_EPOCHS          = 3
BATCH_SIZE          = 1
GRAD_ACCUM_STEPS    = 8
LEARNING_RATE       = 1.5e-4
WARMUP_RATIO        = 0.05
SAVE_DIR            = Path('/kaggle/working/qwen2vl_qlora')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    collate_fn  = collate_fn,
    num_workers = 2,
    pin_memory  = True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    collate_fn  = collate_fn,
    num_workers = 2,
    pin_memory  = True,
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = LEARNING_RATE,
    weight_decay = 0.01,
)

total_steps   = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler     = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f'Optimiser      : AdamW   lr={LEARNING_RATE}')
print(f'Total steps    : {total_steps}')
print(f'Warmup steps   : {warmup_steps}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scaler = torch.cuda.amp.GradScaler(enabled=True)

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

model.train()

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]')

    for step, batch in pbar:
        # Move tensors to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss    = outputs.loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix({'loss': f'{running_loss/(step+1):.4f}',
                          'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

    epoch_train_loss = running_loss / len(train_loader)
    history['train_loss'].append(epoch_train_loss)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Val]'):
            batch    = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                        for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                outputs  = model(**batch)
            val_loss += outputs.loss.item()

    epoch_val_loss = val_loss / len(val_loader)
    history['val_loss'].append(epoch_val_loss)

    print(f'\nEpoch {epoch:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}')

    # ── Save best checkpoint ─────────────────────────────────────────────────
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        model.save_pretrained(SAVE_DIR / 'best_checkpoint')
        processor.save_pretrained(SAVE_DIR / 'best_checkpoint')
        print(f'  ✓ Best model saved  (val_loss = {best_val_loss:.4f})')

# Save final checkpoint
model.save_pretrained(SAVE_DIR / 'final_checkpoint')
processor.save_pretrained(SAVE_DIR / 'final_checkpoint')
print('\nTraining complete ✓')

### 4.6 — Training & Validation Loss Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, history['train_loss'], 'b-o', label='Train Loss',      linewidth=2)
ax.plot(epochs, history['val_loss'],   'r-s', label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('QLoRA Fine-Tuning — Loss Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('/kaggle/working/loss_curves.png', dpi=150)
plt.show()

print(f'Best validation loss : {best_val_loss:.4f}')

## 📝 Part 5 — Markdown Generation on Validation Set

In [ ]:
# ── Load best fine-tuned model ────────────────────────────────────────────────
from peft import PeftModel

fine_tuned_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = 'auto',
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
fine_tuned_model = PeftModel.from_pretrained(
    fine_tuned_model, SAVE_DIR / 'best_checkpoint'
)
fine_tuned_model.eval()
print('Fine-tuned model loaded ✓')

In [ ]:
def generate_markdown(model, processor, image: Image.Image,
                      max_new_tokens: int = MAX_TOKENS) -> str:
    """Run inference: image → markdown string."""
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text',  'text': INSTRUCTION},
            ],
        }
    ]
    text         = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text          = [text],
        images        = image_inputs,
        videos        = video_inputs,
        return_tensors= 'pt',
        padding       = True,
    ).to(device)

    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            generated_ids = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = False,
                temperature    = None,
                repetition_penalty = 1.1,
            )

    # Trim prompt tokens
    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    decoded = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return decoded[0].strip()

print('generate_markdown() helper ready ✓')

In [ ]:
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

scorer  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth  = SmoothingFunction().method1

# ── Generate & score on 5 validation samples ─────────────────────────────────
NUM_VAL_SAMPLES = min(5, len(val_data))
val_results     = []

for i, sample in enumerate(tqdm(val_data[:NUM_VAL_SAMPLES], desc='Generating val')):
    img_path = sample['messages'][0]['content'][0]['image']
    image    = Image.open(img_path).convert('RGB')
    ground   = sample['messages'][1]['content']

    generated = generate_markdown(fine_tuned_model, processor, image)

    # ROUGE
    r_scores = scorer.score(ground, generated)

    # BLEU (token level)
    ref_tok  = nltk.word_tokenize(ground.lower())
    hyp_tok  = nltk.word_tokenize(generated.lower())
    bleu     = sentence_bleu([ref_tok], hyp_tok, smoothing_function=smooth)

    val_results.append({
        'idx'          : i,
        'image'        : image,
        'ground_truth' : ground,
        'generated'    : generated,
        'rouge1'       : r_scores['rouge1'].fmeasure,
        'rouge2'       : r_scores['rouge2'].fmeasure,
        'rougeL'       : r_scores['rougeL'].fmeasure,
        'bleu'         : bleu,
    })

# Summary
df_val = pd.DataFrame([{k: v for k, v in r.items() if k not in ('image','ground_truth','generated')}
                       for r in val_results])
print(df_val[['rouge1','rouge2','rougeL','bleu']].describe().round(4))

## 🔬 Part 5 — Visualization: Ground Truth vs Generated

In [ ]:
def show_comparison(result: dict):
    """Display one image with GT vs Generated side-by-side."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    # Column 1 — image
    axes[0].imshow(result['image'])
    axes[0].set_title('Input Document Image', fontsize=11, fontweight='bold')
    axes[0].axis('off')

    # Column 2 — Ground Truth
    gt_text = result['ground_truth'][:1500]
    axes[1].text(0.01, 0.99, gt_text, transform=axes[1].transAxes,
                 fontsize=6.5, verticalalignment='top', family='monospace',
                 color='darkgreen')
    axes[1].set_title('Ground Truth Markdown', fontsize=11, fontweight='bold')
    axes[1].axis('off')

    # Column 3 — Generated
    gen_text = result['generated'][:1500]
    axes[2].text(0.01, 0.99, gen_text, transform=axes[2].transAxes,
                 fontsize=6.5, verticalalignment='top', family='monospace',
                 color='navy')
    axes[2].set_title('Generated Markdown (Fine-tuned)', fontsize=11, fontweight='bold')
    axes[2].axis('off')

    plt.suptitle(
        f"ROUGE-L: {result['rougeL']:.3f}  |  BLEU: {result['bleu']:.3f}",
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/val_comparison_{result['idx']}.png",
                dpi=120, bbox_inches='tight')
    plt.show()


for r in val_results:
    show_comparison(r)

## 🧪 Part 6 — Testing: 3 Training + 3 Unseen Images

In [ ]:
# ── 3 samples from training set ───────────────────────────────────────────────
train_test_samples = random.sample(train_data, 3)

print('=' * 60)
print('SECTION A  —  3 Training Images')
print('=' * 60)

for i, sample in enumerate(train_test_samples):
    img_path  = sample['messages'][0]['content'][0]['image']
    image     = Image.open(img_path).convert('RGB')
    ground    = sample['messages'][1]['content']
    generated = generate_markdown(fine_tuned_model, processor, image)

    r_scores  = scorer.score(ground, generated)
    print(f'\n--- Training Sample {i+1} ---')
    print(f'  ROUGE-L  : {r_scores["rougeL"].fmeasure:.4f}')
    print(f'  Generated (first 400 chars):\n{generated[:400]}')
    print(f'  GT        (first 400 chars):\n{ground[:400]}')

    show_comparison({
        'idx': f'train_{i+1}', 'image': image,
        'ground_truth': ground, 'generated': generated,
        'rougeL': r_scores['rougeL'].fmeasure, 'bleu': 0.0,
    })

In [ ]:
# ── 3 UNSEEN document images (user-provided or downloaded) ────────────────────
#
# Place 3 unseen document images in /kaggle/working/unseen_docs/ before running.
# They should be scanned documents, research paper pages, or form screenshots.

UNSEEN_DIR = Path('/kaggle/working/unseen_docs')
UNSEEN_DIR.mkdir(exist_ok=True)

unseen_imgs = sorted(
    [p for p in UNSEEN_DIR.iterdir() if p.suffix.lower() in image_extensions]
)[:3]

if not unseen_imgs:
    print('⚠  No unseen images found in /kaggle/working/unseen_docs/')
    print('   Add 3 document images (.png / .jpg) and rerun this cell.')
else:
    print('=' * 60)
    print('SECTION B  —  3 Unseen Document Images')
    print('=' * 60)

    for i, img_path in enumerate(unseen_imgs):
        image     = Image.open(img_path).convert('RGB')
        generated = generate_markdown(fine_tuned_model, processor, image)

        print(f'\n--- Unseen Sample {i+1} : {img_path.name} ---')
        print(f'  Generated (first 500 chars):\n{generated[:500]}')

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        axes[0].imshow(image); axes[0].axis('off')
        axes[0].set_title(f'Unseen Image {i+1}: {img_path.name}', fontweight='bold')
        axes[1].text(0.01, 0.99, generated[:1500], transform=axes[1].transAxes,
                     fontsize=6.5, verticalalignment='top', family='monospace',
                     color='navy')
        axes[1].set_title('Generated Markdown', fontweight='bold')
        axes[1].axis('off')
        plt.tight_layout()
        plt.savefig(f'/kaggle/working/unseen_result_{i+1}.png', dpi=120, bbox_inches='tight')
        plt.show()

## 📊 Bonus — Zero-shot vs Fine-tuned Comparison

In [ ]:
# ── Load base (zero-shot) model ───────────────────────────────────────────────
base_zs = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = 'auto',
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
base_zs.eval()

test_sample = val_data[0]
img_path    = test_sample['messages'][0]['content'][0]['image']
image       = Image.open(img_path).convert('RGB')
ground      = test_sample['messages'][1]['content']

zs_output   = generate_markdown(base_zs,          processor, image)
ft_output   = generate_markdown(fine_tuned_model, processor, image)

zs_rouge = scorer.score(ground, zs_output)['rougeL'].fmeasure
ft_rouge = scorer.score(ground, ft_output)['rougeL'].fmeasure

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(image); axes[0].axis('off')
axes[0].set_title('Input Image', fontweight='bold')

axes[1].text(0.01, 0.99, zs_output[:1400], transform=axes[1].transAxes,
             fontsize=6.5, verticalalignment='top', family='monospace', color='gray')
axes[1].set_title(f'Zero-Shot Output\nROUGE-L = {zs_rouge:.3f}', fontweight='bold')
axes[1].axis('off')

axes[2].text(0.01, 0.99, ft_output[:1400], transform=axes[2].transAxes,
             fontsize=6.5, verticalalignment='top', family='monospace', color='navy')
axes[2].set_title(f'Fine-Tuned Output\nROUGE-L = {ft_rouge:.3f}', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Zero-Shot vs QLoRA Fine-Tuned — Output Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/zeroshot_vs_finetuned.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Zero-shot ROUGE-L  : {zs_rouge:.4f}')
print(f'Fine-tuned ROUGE-L : {ft_rouge:.4f}')
print(f'Improvement        : {(ft_rouge - zs_rouge)*100:+.2f} pp')

## 🌐 App Deployment — Gradio

In [ ]:
import gradio as gr

def predict(image_input):
    """Gradio handler: PIL image → markdown string."""
    if image_input is None:
        return '⚠ Please upload a document image.'
    img = Image.fromarray(image_input).convert('RGB')
    md  = generate_markdown(fine_tuned_model, processor, img)
    return md

with gr.Blocks(theme=gr.themes.Soft(), title='Doc → Markdown') as demo:
    gr.Markdown(
        '# 📄 Document → Markdown Generator\n'
        'Upload a document image and the fine-tuned **Qwen2-VL** model will '
        'convert it to structured Markdown.\n\n'
        '_AI4009 — Assignment 5 | 22F-8816_'
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_in  = gr.Image(label='📷 Upload Document Image', type='numpy')
            btn       = gr.Button('Generate Markdown ✨', variant='primary')

        with gr.Column(scale=1):
            md_out    = gr.Textbox(
                label='📝 Generated Markdown',
                lines=25,
                placeholder='Markdown output will appear here …',
            )
            rendered  = gr.Markdown(label='🖥 Rendered Preview')

    btn.click(fn=predict, inputs=image_in, outputs=[md_out])
    md_out.change(fn=lambda x: x, inputs=md_out, outputs=rendered)

    gr.Examples(
        examples  = [[str(p)] for p in sorted(DATASET_ROOT.rglob('*.png'))[:3]],
        inputs    = image_in,
        label     = 'Example Images from Nougat Dataset',
    )

demo.launch(share=True, debug=False)
print('Gradio app launched ✓  Check the public URL above.')

---
## ✅ Summary

| Component | Detail |
|---|---|
| Base model | `Qwen2-VL-2B-Instruct` |
| Fine-tuning method | QLoRA (4-bit NF4 + LoRA rank 16) |
| Dataset | Nougat Training Dataset Example (Kaggle) |
| Split | 80% Train / 20% Validation |
| Epochs | 3 |
| Batch size | 1 + gradient accumulation 8 (effective batch = 8) |
| Learning rate | 1.5e-4 (cosine schedule + warmup 5%) |
| Image resolution | 640 px (longest edge) |
| Metrics | ROUGE-1/2/L, BLEU |
| Deployment | Gradio (image upload → markdown) |
| Platform | Kaggle GPU T4 × 2 |

---
*Assignment 5 | AI4009 Generative AI | FAST-NUCES Spring 2026 | Roll: 22F-8816*